In [0]:
dbutils.widgets.text("catalog","fk_1392")
catalog = dbutils.widgets.get("catalog")

#stores

In [0]:
from pyspark.sql.functions import *

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load(f"/Volumes/{catalog}/landing/raw/stores/stores.csv")
    
df = df.withColumn("ingestion_timestamp",current_timestamp()) \
    .withColumn("source_file",col("_metadata.file_path"))

df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.stores")

#customers

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.bronze.customers (
  customer_id STRING,
  full_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  tier STRING,
  signup_date STRING
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")



In [0]:
spark.sql(f"""
COPY INTO {catalog}.bronze.customers
FROM "/Volumes/{catalog}/landing/raw/customers/"
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")

#products

In [0]:
# CATEGRY - OBJECT L1,L2
# pack - object
# mrp - number
# selling_price - number
# is_active boolean
# tags - array of strings
# snapshot of date string

spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.bronze.products (
        product_id STRING,
        product_name STRING,
        brand STRING,
        category STRUCT<
            l1: STRING,
            l2: STRING
        >,
        pack STRUCT<
            size: STRING,
            unit: STRING
        >,
        mrp DOUBLE,
        selling_price STRING,
        is_active STRING,
        tags ARRAY<STRING>,
        snapshot_date STRING,
        ingestion_timestamp TIMESTAMP,
        source_file STRING
         

        
    ) USING DELTA
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")

spark.sql(f"ALTER TABLE {catalog}.bronze.products ALTER COLUMN ingestion_timestamp SET DEFAULT current_timestamp()")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.bronze.products (
    product_id STRING,
    product_name STRING,
    brand STRING,
    category STRUCT<l1: STRING, l2: STRING>,
    pack STRUCT<size: LONG, unit: STRING>,
    mrp DOUBLE,
    selling_price DOUBLE,
    is_active BOOLEAN,
    tags ARRAY<STRING>,
    snapshot_date STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
) USING DELTA
""")

spark.sql(f"""
COPY INTO {catalog}.bronze.products
FROM "/Volumes/{catalog}/landing/raw/products/"
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiline' = 'true')
""")

#orders

In [0]:
# ORDER_ID -String
# CUSTOMER_ID -String
# store_id - string
#order_ts - string
#updated_at - string
#order_status - string
#payment - object - method, txn_ref
#delivery - object - promised_mins,actural_mins, delivery_fee
#iems - array of objects - product_id,unit_ptice,discount(optionally)
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.bronze.orders (
    order_id STRING,
    customer_id STRING,
    store_id STRING,
    order_ts STRING,
    updated_at STRING,
    order_status STRING,
    payment STRUCT<
        method: STRING,
        txn_ref: STRING
    >,
    delivery STRUCT<
        promised_mins: LONG,
        actual_mins: LONG,
        delivery_fee: DOUBLE
    >,
    items ARRAY<STRUCT<
        product_id: STRING,
        unit_price: DOUBLE,
        discount: DOUBLE
    >>,
        ingestion_timestamp TIMESTAMP,
    source_file STRING
    
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")

spark.sql(f"ALTER TABLE {catalog}.bronze.orders ALTER COLUMN ingestion_timestamp SET DEFAULT current_timestamp()")

In [0]:
# spark.sql(f"""
# COPY INTO {catalog}.bronze.orders
# FROM "/Volumes/{catalog}/landing/raw/orders/"
# FILEFORMAT = JSON
# FORMAT_OPTIONS ('multiline' = 'true')
# COPY_OPTIONS ('mergeSchema' = 'true')
# """)


In [0]:
from pyspark.sql.functions import *

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/landing/raw/orders/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("multiLine", "true") \
    .load(f"/Volumes/{catalog}/landing/raw/orders/")

df = df.withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path"))



In [0]:
df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/checkpoint_orders_v2") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .table(f"{catalog}.bronze.orders") 